# 02 · 이미 있는 zarr → ckpt

**위에서 아래로 한 번씩.** 앞 셀이 뒤 셀에 필요한 걸 전부 만든다 — 건너뛸 자리가 없다.

zarr 를 새로 만드는 건 `01_slam_to_ckpt.ipynb` 다. 이 노트북은 **zarr 가 이미 있을 때** 쓴다.

## 0. 설정 — 여기만 고친다

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path

# ── 여기 네 줄만 고친다 ──────────────────────────────────────
ZARR   = Path("/home/j-j15a103/hyeonseok/umi_gpu_bundle/data/s22_pick_20260921_atlas_v1/s22_pick_20260921_atlas_v1.zarr.zip")
NAME   = "atlas_v1_ours"     # 산출물 폴더 이름. 남의 결과를 덮지 않게 고유하게
EPOCHS = 120
GPU    = "1"                 # 1·2·3·4·6 만. 0·5·7·8·9 는 타인 것
# ────────────────────────────────────────────────────────────

BATCH   = 64
RATE_HZ = 30.0
HOME    = Path.home()
REPO    = HOME / "S15P21A103"
BUNDLE  = HOME / "hyeonseok" / "umi_gpu_bundle"     # 읽기만 한다. 아무것도 쓰지 않는다
PY_GATE = HOME / "envs/handoff312/bin/python"       # 계측기·export·검사
PY_TRN  = "python"                                  # 학습기 (시스템 파이썬, torch cu126)
OUT     = REPO / "out" / NAME
OUT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, env=None, title=None):
    """명령을 돌리며 출력을 **실시간으로** 흘린다. 반환은 종료코드."""
    if title: print(f"── {title}\n$ {' '.join(map(str, cmd))}\n", flush=True)
    e = dict(os.environ); e.update(env or {})
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd and str(cwd), env=e,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait(); print(f"\n[종료코드 {p.returncode}]", flush=True)
    return p.returncode

def jload(p): return json.loads(Path(p).read_text(encoding="utf-8"))

print("설정 —")
for k, v in [("ZARR", ZARR), ("REPO", REPO), ("BUNDLE", BUNDLE), ("OUT", OUT)]:
    print(f"  [{'있음' if Path(v).exists() else '없음'}] {k:7} {v}")
print(f"\n  NAME {NAME} · EPOCHS {EPOCHS} · BATCH {BATCH} · GPU {GPU}")
assert ZARR.exists(), f"zarr 가 없다: {ZARR}"
print(f"  zarr 크기 {ZARR.stat().st_size:,} B")

## 1. 프리플라이트 — 가용성이 아니라 **기능**을 본다

GPU 는 실제 행렬곱까지 시킨다. `torch.cuda.is_available()` True 인데 커널이 없어
학습이 죽은 실증이 있다 🟢

In [ ]:
ok = tot = 0
def chk(name, cond, hint=""):
    global ok, tot
    tot += 1; ok += bool(cond)
    print(f"  [{'있음' if cond else '없음'}] {name}" + ("" if cond else f"   <- {hint}"))

chk("게이트용 python", PY_GATE.exists(), str(PY_GATE))
chk("학습기", (BUNDLE/"03_umi_policy_trainer/train_policy.py").exists(), str(BUNDLE))
chk("학습 profile", (BUNDLE/"03_umi_policy_trainer/configs/policy_resnet18_gpu.yaml").exists(), "")
chk("공식 UMI", (BUNDLE/"third_party/umi").exists(), "")
for t in ("audit_umi_zarr", "export_deploy_ckpt", "smoke_deploy_ckpt"):
    chk(f"계측기 {t}", (REPO/"AI/tools"/f"{t}.py").exists(), "git pull origin ai")
print(f"\n필수 {ok} / {tot}")
assert ok == tot, "위 [없음] 을 먼저 해결한다"

import textwrap
rc = run([PY_TRN, "-c", textwrap.dedent("""
    import torch
    print("torch", torch.__version__, "available", torch.cuda.is_available())
    assert torch.cuda.is_available(), "CUDA 없음"
    a = torch.randn(512, 512, device="cuda")
    print("device", torch.cuda.get_device_name(0), "cc", torch.cuda.get_device_capability(0))
    print("실연산 OK", round((a @ a).sum().item(), 3))
""")], env={"CUDA_VISIBLE_DEVICES": GPU}, title="GPU 실연산")
assert rc == 0, "GPU 가 안 된다. 학습으로 넘어가지 않는다"

## 2. zarr 감사 — **막지 않는다.** 무슨 데이터인지 기록만 한다

게이트 FAIL 이어도 학습은 돈다. 불합격 항목이 무엇인지 알고 넘어가는 게 목적이다.
🟢 atlas_v1 참조: 통과 5 · 불합격 4 (편 길이 218 · 경로/직선 2.89 · EEF 0.466m · 회전 0.749rad)

In [ ]:
rc = run([PY_GATE, REPO/"AI/tools/audit_umi_zarr.py", "--zarr", ZARR,
          "--label", NAME, "--gate", "--rate-hz", RATE_HZ,
          "--out", OUT/"zarr_gate.json"], title="zarr 감사 + 게이트")
if rc == 0 and (OUT/"zarr_gate.json").exists():
    gg = jload(OUT/"zarr_gate.json").get("gates", {})
    print(f"\n>> {gg.get('verdict')} · 통과 {gg.get('passed')} · 불합격 {gg.get('failed')} "
          f"· 미판정 {gg.get('unknown')} / {gg.get('total')}")
    for r in gg.get("rows", []):
        if r.get("ok") is not True:
            print("   미통과", r["name"], r["got"], "기준", r["want"])
else:
    print("\n** 감사가 못 돌았다. 학습은 진행할 수 있지만 데이터 조건을 모르는 채로 간다 **")

## 3. 학습

**평가 지표는 롤아웃 성공률이다. train/val loss 가 아니다.**
val 0.00547→0.00517 인데 롤아웃 0% 그대로였던 실증이 있다 (독립 2회) 🟢

120 epoch · 70편 기준 약 70분. 이 셀이 로그를 실시간으로 흘린다.

In [ ]:
RUN_ID  = f"{NAME}_{time.strftime('%H%M%S')}"
TRAINER = BUNDLE / "03_umi_policy_trainer" / "train_policy.py"
PROFILE = BUNDLE / "03_umi_policy_trainer" / "configs" / "policy_resnet18_gpu.yaml"
env = {"CUDA_VISIBLE_DEVICES": GPU, "UMI_ROOT": str(BUNDLE/"third_party/umi"),
       "WANDB_MODE": "disabled", "HYDRA_FULL_ERROR": "1"}
print("RUN_ID", RUN_ID)

rc = run([PY_TRN, TRAINER, "check", ZARR], cwd=BUNDLE, env=env, title="dataset check")
assert rc == 0, "dataset check 실패 — 학습으로 넘어가지 않는다"

rc = run([PY_TRN, TRAINER, "train", ZARR, "--run-id", RUN_ID, "--runs-dir", OUT/"runs",
          "--epochs", EPOCHS, "--batch", BATCH, "--eval-batch", BATCH, "--profile", PROFILE],
         cwd=BUNDLE, env=env, title=f"학습 {EPOCHS} epoch")
assert rc == 0, "학습 실패"

MAN = OUT/"runs"/RUN_ID/"manifest.json"
CK  = OUT/"runs"/RUN_ID/"checkpoints"/"best.ckpt"
# 종료코드 0 만 보면 '성공했는데 파일이 없다' 가 통과한다
assert MAN.exists() and CK.exists(), f"종료코드 0 인데 산출물이 없다: {MAN} · {CK}"

m = jload(MAN); b = m["best_checkpoint"]
p, h = b["policy"], b["hold_current_pose_and_width"]
print(f"\nepoch {b['epoch']} · baseline 이김 {b['beats_hold_baseline']}\n")
print(f"{'지표':34}{'정책':>12}{'hold':>12}{'배수':>8}")
for k in ("position_component_rmse_mm","position_distance_rmse_mm","position_distance_p95_mm",
          "rotation_mean_deg","width_rmse_mm"):
    print(f"{k:34}{p[k]:>12.3f}{h[k]:>12.3f}{h[k]/p[k]:>7.2f}x")
print("\n** 개루프 예측 오차다. 롤아웃 성공률이 아니다 **")
print("참조 🟢 현석 atlas_v1 e120 3.67x · s22_pick_v3 3.12x · v4 3.52x")

## 4. 배포용 export → 배포 검사

**학습 ckpt 를 검사기에 바로 넣지 않는다.** 학습 매니페스트에는 배포 계약
(`actionSpec` · `nParams` · `sha256_export` · `chunk_anchor`)이 없다.
그걸 만드는 게 `export_deploy_ckpt.py` 다 — **export 다음에 검사**다 (2026-09-21 실증 🟢).

In [ ]:
DCK  = OUT/"runs"/RUN_ID/"deploy"/f"{RUN_ID}.ckpt"
DMAN = DCK.with_suffix(".manifest.json")     # export 가 이 이름으로 쓴다

rc = run([PY_GATE, REPO/"AI/tools/export_deploy_ckpt.py",
          "--checkpoint", CK, "--out", DCK, "--note", f"{NAME} {RUN_ID}"],
         cwd=REPO, title="배포용 export (ema_model 만 남기고 계약 manifest 작성)")
assert rc == 0, "export 실패"
assert DCK.exists() and DMAN.exists(), f"종료코드 0 인데 산출물이 없다: {DCK} · {DMAN}"

rc = run([PY_GATE, REPO/"AI/tools/smoke_deploy_ckpt.py",
          "--checkpoint", DCK, "--manifest", DMAN], cwd=REPO, title="ckpt 배포 검사")
print("\n배포 ckpt:", DCK)
print("다음 — 실물 투입은 RUNBOOK 6절. --dry-run -> 물체 없이 -> 본 실행 순서.")

## 한계 — 먼저 말한다

- 위 배수는 **개루프 예측 오차**다. 롤아웃 성공률이 아니고 실물 성공률은 더더욱 아니다
- 같은 설정으로도 **학습 실행 간 편차가 있다** 🟢. 1회 실행은 판정이 아니라 방향이다
- `camera_tcp` 가 `physically_validated` 가 아니면 `physical_deployment_ready: false` 로 남는다.
  그 상태 수치는 조건 병기 없이 인용하지 않는다
- 실물 롤아웃 불가 (모터 고장). 시뮬·개루프 지표만 나온다